# Simple StoryMap Creator

In [1]:
from arcgis.gis import GIS
gis = GIS(
    url="https://mapit.mapsdevext.arcgis.com/",
    username="wponcy_dev",
    password="Youstayforme2026!"
)
from arcgis.apps.storymap import StoryMap, story, story_content
from arcgis.apps.storymap.story_content import Image, TextStyles, Video, Audio, Embed, Map, Cover, Text, Button, Gallery, Swipe, Sidecar, Timeline
import arcgis.geometry as arcGeo
import time

Function to create StoryMaps for a Showcase's web map

In [2]:
class webMap:
    def __init__(self, itemId, geometry):
                # creating a union of the geometries which will be the centerpoint of our map 

        # gathering details from the item
        self.itemId = itemId
        self.item = gis.content.get(item_id)
        self.title = self.item.title
        self.thumbnail_path = self.item.download_thumbnail()
        self.thumbnail = Image(self.thumbnail_path)
        self.summary = self.item.snippet
        self.description = self.item.description
        self.geometry = arcGeo.union(all_geometries, spatial_ref = srs)[0] 
        self.geometryExtent = self.geometry.extent
    
    
def createStoryMapforWebMap(webMapObject):

    print('Creating the empty StoryMap')
    # creating an empty storymap
    new_story = StoryMap()

    # filling in the header & byline
    
    new_story.contents[0].title = webMapObject.title
    new_story.contents[0].summary = 'Made with the ArcGIS API for Python'
    new_story.contents[0].date = 'current-date'
    
    # adding an image to the storymap
    print('Adding an image')
    new_node = new_story.add(webMapObject.thumbnail, position = 2)
    
    print('Adding a map')
    my_map = Map(webMapObject.itemId)
    new_node = new_story.add(my_map, webMapObject.title)
    
    print('Setting focus area extent to:', webMapObject.geometry.extent)
    my_map.set_viewpoint(extent={
        'spatialReference': {'latestWkid': 3857, 'wkid': 102100},
        'xmin':  webMapObject.geometry.extent[0], # x-min
        'ymin':  webMapObject.geometry.extent[1], # y-min
        'xmax':  webMapObject.geometry.extent[2], # x-max,
        'ymax':   webMapObject.geometry.extent[3], # y-max
    })

    # adding credits
    new_story.credits("StoryMaps" , "Python API", "Thank You for Reading","Esri Living Atlas of the World")
    
    print('Saving the StoryMap')
    storyMapTitle = f'StoryMap for {webMapObject.title}'
    new_story.save(title=storyMapTitle, access='public')
    
    print('Updating the item details')
    print(updateStoryMapItemDetails(storyMapTitle, webMapObject.description, webMapObject.thumbnail_path))

def updateStoryMapItemDetails(SMtitle, WMdescription, WMthumbnail, retries=5, delay=5):
    # time.sleep(20)

    for attempt in range(1, retries):
        story_item_search = gis.content.search(query=f'title:"{SMtitle}"',item_type="StoryMap")
        if story_item_search:
            story_item = story_item_search[0]
            try: 
                # then change the description
                update_status = story_item.update(
                    item_properties={
                        "access": "public",
                        "description": str(WMdescription),
                        "snippet": "This StoryMap was made with the ArcGIS API for Python",
                    },
                    thumbnail = WMthumbnail # updating thumbnail
                )

                if update_status:
                    return("Successfully updated the StoryMap.")
                else: 
                    return("Failure updating StoryMap.")
        
            except Exception as e:
                return(f"An unexpected error occurred: {e}")
        else:
            waitTime = attempt * delay
            print(f'No item found with the query "query="title:{SMtitle}", item_type="StoryMap", waiting {waitTime} seconds.')
            time.sleep(waitTime)

showcaseID = input('Paste in an item ID for a Showcase item.')
showcase = gis.content.get(showcaseID)
all_showcase_data = showcase.get_data()
for item in all_showcase_data['items']:
    
    if item['type'] == "Web Map":

        item_id = item['id']
        
        # creating a union of the geometries 
        all_geometries = []
        for geometry in item['highlightedArea']['graphics']:
            srs = geometry['geometry']['spatialReference']
            current_geometry = arcGeo.Geometry(geometry['geometry'])
            all_geometries.append(current_geometry)
        union = arcGeo.union(all_geometries, spatial_ref = srs)[0]

        print('Creating a webmap for id', item_id)
        currentWebMap = webMap(item_id, union)
        
        print(f'Creating StoryMap for item id: {currentWebMap.title}')
        createStoryMapforWebMap(currentWebMap)
        print('-' * 50)

Paste in an item ID for a Showcase item. 59328dd9074e4b17a3a4807ab752f2ad


Creating a webmap for id 7d218a4ccee54bf8888d107fbceb8512
Creating StoryMap for item id: Monthly Soil Moisture
Creating the empty StoryMap
Adding an image
Adding a map
Setting focus area extent to: (-13849814.4428, 3833635.0588999987, -12705028.403099999, 5162404.93)
Saving the StoryMap
Updating the item details
No item found with the query "query="title:StoryMap for Monthly Soil Moisture", item_type="StoryMap", waiting 5 seconds.
Successfully updated the StoryMap.
--------------------------------------------------
Creating a webmap for id d6fd1e9992fa42c8bcdd40927c4e6f1c
Creating StoryMap for item id: Monthly Snow Pack
Creating the empty StoryMap
Adding an image
Adding a map
Setting focus area extent to: (-13849814.4428, 3833635.0588999987, -12705028.403099999, 5162404.93)
Saving the StoryMap
Updating the item details
Successfully updated the StoryMap.
--------------------------------------------------
Creating a webmap for id 8942e49021ee4cfba105a19ed03a7203
Creating StoryMap for ite

Showcase example ID: 59328dd9074e4b17a3a4807ab752f2ad